# Lesson 1.2 — Model Options: Fast, Extended, and Adaptive Thinking

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish fast mode, extended thinking, and adaptive thinking / effort levels
- Enable extended thinking with a token budget via the Messages API
- Decide when added reasoning is worth the latency and cost
- Read thinking blocks from a response safely

**You will produce:** A script that toggles extended thinking and inspects thinking vs text blocks.

## Concepts

Claude can answer directly (**fast mode**) or spend extra internal reasoning before replying (**extended thinking**). Some models support **adaptive thinking**, deciding how much to think, and expose **effort** controls to trade quality for speed/cost.

- Enable extended thinking with a `thinking` config and a `budget_tokens` allowance. The model emits **thinking** content blocks plus the final **text** block.
- Use extended thinking for multi-step reasoning, math, planning, and hard tool decisions; skip it for simple lookups where latency matters.
- Always iterate over `response.content` by block `type` - do not assume index 0 is text when thinking is enabled.

> Exact parameter names and supported models evolve; confirm the current `thinking` schema and effort levels at https://docs.anthropic.com before production use.

> **Reference:** [Anthropic - Extended thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A fast (no-thinking) call

Baseline: a direct answer with no extended reasoning.

In [ ]:
question = "A train leaves at 2pm going 60mph. How far in 3.5 hours?"

def fast_answer(q):
    if not HAVE_API:
        return mock_text(q)
    msg = client.messages.create(
        model=MODEL_SONNET, max_tokens=200,
        messages=[{"role": "user", "content": q}],
    )
    return msg.content[0].text

print(fast_answer(question))

## Code-Along: Step 2 — Enable extended thinking

Add a `thinking` block with a token budget. We parse blocks by type so thinking content is handled separately from the answer.

In [ ]:
def thinking_answer(q, budget=1024):
    if not HAVE_API:
        return "[MOCK] thinking=...", mock_text(q)
    msg = client.messages.create(
        model=MODEL_SONNET,
        max_tokens=2048,
        thinking={"type": "enabled", "budget_tokens": budget},
        messages=[{"role": "user", "content": q}],
    )
    thinking, text = "", ""
    for block in msg.content:
        if block.type == "thinking":
            thinking += block.thinking
        elif block.type == "text":
            text += block.text
    return thinking, text

thinking, text = thinking_answer(question)
print("THINKING (truncated):", thinking[:200])
print("ANSWER:", text)

## Exercise

Write `choose_mode(task_description)` that returns `'fast'` or `'thinking'` based on simple heuristics (e.g., presence of words like 'plan', 'prove', 'multi-step', 'debug'). Justify each rule in a comment.

In [ ]:
def choose_mode(task_description):
    # TODO: return 'thinking' for reasoning-heavy tasks, else 'fast'
    pass

# TODO: test on 'look up a capital city' and 'plan a 5-step migration'

## Key Takeaways

- Fast mode answers directly; extended thinking adds internal reasoning for a token budget.
- Enable thinking via a thinking config with budget_tokens and parse blocks by type.
- Reserve extended/adaptive thinking for multi-step reasoning where quality beats latency.
- Never assume content[0] is text - iterate blocks when thinking is on.

---

**Next:** [Lesson 1.3 — Prompting Techniques: Zero-, Single-, and Multi-Shot](./lesson-1.3-prompting-shots.ipynb)